# Assignment 8 — Person 2: completed RAG evaluations and scaling

This notebook analyzes the returned Google Colab CPU run and joins real generated answers, retrieved chunks, measured request latency and AI-assisted reviews under Person 3's frozen rubric. The 15-row matrix and 10-row scaling comparison use the same Qwen2-0.5B generator, prompt, greedy decoding, output cap, small encoder and top-k. Student signoff on assisted judgments is not recorded.

Historical retrieval-only evidence remains under `results/person2/small-cpu-2026-11-08/`; completed RAG evidence is separate. This is the Person 2 contribution for integration into the team's final notebook, report and ZIP.

In [1]:
from pathlib import Path
import sys,json
import pandas as pd
from IPython.display import display,Markdown
ROOT=Path.cwd()
if not (ROOT/'a8/corpus.py').exists() and ROOT.name=='a8': ROOT=ROOT.parent
assert (ROOT/'a8/corpus.py').exists(), 'Start from repository root or a8/'
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from a8.person2_analysis import DEFAULT,evaluation_table,export_results
from a8.person2_experiments import read_rows
from a8.corpus import load_queries,load_paragraphs,load_chunks,word_count,STRATEGIES
from a8.retrieval import MODELS
RUN_DIR=ROOT/'a8/results/person2/completed-2026-11-09'
# Saved-evidence mode requires no downloads or model weights.
rows=evaluation_table(RUN_DIR)
assert len(rows)==25
frame=pd.DataFrame(rows)
print('Validated 15 matrix and 10 scaling rows, frozen controls, prompts, corpus and full-row review hashes.')
print('Review method: AI-assisted; student signoff not recorded.')

Validated 15 matrix and 10 scaling rows, frozen controls, prompts, corpus and full-row review hashes.
Review method: AI-assisted; student signoff not recorded.


## Data, model choices and frozen generation contract

In [2]:
display(pd.DataFrame([dict(size=size,**spec) for size,spec in MODELS.items()]))
display(pd.DataFrame([dict(dataset=ds,paragraphs=len(load_paragraphs(ds)),
    words=sum(word_count(p.text) for p in load_paragraphs(ds)),
    **{s:len(load_chunks(s,ds)) for s in STRATEGIES}) for ds in ('small','full')]))
manifest=json.loads((RUN_DIR/'matrix/manifest.json').read_text())
display(pd.DataFrame([{'generator':manifest['generator'],'revision':manifest['revision'],
    'decoding':manifest['decoding'],'max_new_tokens':manifest['max_new_tokens'],'top_k':manifest['top_k']}]))
print('Execution details:',manifest['environment'])

,size,model_id,dimension,license,max_seq_length,query_prefix
0,small,sentence-transformers/all-MiniLM-L6-v2,384,Apache-2.0,256,
1,medium,sentence-transformers/all-mpnet-base-v2,768,Apache-2.0,384,
2,large,BAAI/bge-large-en-v1.5,1024,MIT,512,Represent this sentence for searching relevant...


,dataset,paragraphs,words,fixed,overlap,hybrid
0,small,15,1461,15,10,10
1,full,30,2901,30,20,20


,generator,revision,decoding,max_new_tokens,top_k
0,Qwen/Qwen2-0.5B-Instruct,c540970f9e29518b1d8f06ab8b24cba66ad77b6d,"greedy, 1 beam, float32 CPU",128,3


Execution details: {'torch_threads': 4, 'python': '3.13.18', 'platform': 'Linux-6.6.159+-x86_64-with-glibc2.39', 'machine': 'x86_64', 'packages': {'numpy': '2.4.6', 'torch': '2.14.1+cpu', 'sentence-transformers': '5.1.2', 'transformers': '4.57.6', 'huggingface-hub': '0.36.2'}, 'git_commit': None}


## Fifteen assessed small-model rows

Context quality is exact supporting-evidence coverage (0–100), not passage precision. Answer quality follows the shared required-fact and unsupported-instruction rubric. Latency is observed warmed end-to-end query embedding through decoded answer, excluding model loading, indexing and review. Noise is a separate AI-assisted relevance judgment, not inferred from paragraph IDs.

In [3]:
matrix=frame[frame.experiment=='matrix']
pd.set_option('display.max_colwidth',None)
pd.set_option('display.max_columns',None)
display(matrix[['embedding_model','strategy','query_id','query','retrieved_context_quality',
    'answer_quality','retrieval_ms','generation_ms','end_to_end_ms','notes']])

,embedding_model,strategy,query_id,query,retrieved_context_quality,answer_quality,retrieval_ms,generation_ms,end_to_end_ms,notes
0,sentence-transformers/all-MiniLM-L6-v2,fixed,Q1,"I finished closing the kitchen, but the dishwasher is still running. What should I mark on the board and what should I do with the dishwasher?",100.0,66.67,86.089021,29862.879253,29949.150205,"Completes the existing closing card and assigns unloading to the morning resident. Omits attaching the Running tag. Rank 3 concerns appliance upkeep, not the running dishwasher handoff."
1,sentence-transformers/all-MiniLM-L6-v2,fixed,Q2,The store ran out of an item on our shared grocery list and nobody approved a replacement. What should I record?,100.0,0.00,54.226600,11467.331860,11521.662689,Neither leaves the original unchecked nor writes Out of stock. Closing the entry without an obtained item contradicts the no-replacement case. Rank 3 is a shop-return/refund workflow.
2,sentence-transformers/all-MiniLM-L6-v2,fixed,Q3,"I need the shared cordless drill this afternoon. Where do I find it, what do I log, and where does the kit go when I finish?",100.0,0.00,54.017444,36288.608385,36342.723434,"Omits the blue bin, complete borrower/tool/expected-return record, drill-and-bits return and closure. Unsupported directions: use checkout card as the location source, log a check time, and log invented location/status fields. Appliance consumables at rank 3 are unrelated."
3,sentence-transformers/all-MiniLM-L6-v2,fixed,Q4,The bathroom tap has a slow drip even with the handle shut. How do I report it to our household?,100.0,0.00,41.424276,39983.064480,40024.632925,"Names Plumbing on the Maintenance board, but does not direct reuse of an existing card or record all three report details. Photo is made mandatory and notification is conditioned on an unavailable card; two unsupported instructions. Output cap cuts off the last step. Rank 3 is appliance servicing."
4,sentence-transformers/all-MiniLM-L6-v2,fixed,Q5,A friend is staying overnight. What needs to be in place before I mark their room ready?,100.0,20.00,37.882141,25888.567141,25926.573542,A clean sheet set conveys one set. Towels/carafe lack their required placements; lamp check and the conditional Guest Room Ready completion rule are absent. Departure/review context at rank 3 is unrelated to readiness.
5,sentence-transformers/all-MiniLM-L6-v2,overlap,Q1,"I finished closing the kitchen, but the dishwasher is still running. What should I mark on the board and what should I do with the dishwasher?",100.0,100.00,35.230797,41076.742072,41112.127634,All three required handoff actions appear despite first-person wording and awkward left it unloading phrasing. Appliance-upkeep context at rank 3 is unrelated.
6,sentence-transformers/all-MiniLM-L6-v2,overlap,Q2,The store ran out of an item on our shared grocery list and nobody approved a replacement. What should I record?,100.0,0.00,35.707337,14612.830686,14648.634584,"Quantity still needed is supported background but supplies neither required out-of-stock action. Rank 3 inventory text directly reinforces that any substitution follows approval on the actual shopping entry, so it is relevant conditional context rather than noise."
7,sentence-transformers/all-MiniLM-L6-v2,overlap,Q3,"I need the shared cordless drill this afternoon. Where do I find it, what do I log, and where does the kit go when I finish?",100.0,0.00,34.542153,33512.464598,33547.190222,"No complete required fact: borrower name, exact bin, drill-and-bits return and entry closure are absent. Wrong location source and invented appliance-upkeep return destination are two unsupported instructions. Rank 3 is the unrelated appliance context the answer misuses."
8,sentence-transformers/all-MiniLM-L6-v2,overlap,Q4,The bathroom tap has a slow drip even with the handle shut. How do I report it to our household?,100.0,75.00,49.868729,36946.307874,36996.293516,"Correctly reuses the card, records room/time/closed-handle sta

## Complete retrieved chunks and actual answers

In [4]:
for experiment in ('matrix','scaling'):
    for row in read_rows(RUN_DIR/experiment/'rows.jsonl'):
        print('\n'+'='*72)
        print(experiment,row['row_id'],row['query'])
        for hit in row['retrieved_chunks']:
            print(f"\nRank {hit['rank']} | {hit['chunk_id']} | cosine={hit['score']:.4f}")
            print('Paragraphs:',', '.join(hit['paragraph_ids']))
            print(hit['text'])
        print('\nActual generated answer:',row['answer'])
        print('Output tokens:',row['output_tokens'],'Reached 128-token cap:',row['reached_token_cap'])


matrix small/full/fixed/Q1 I finished closing the kitchen, but the dishwasher is still running. What should I mark on the board and what should I do with the dishwasher?

Rank 1 | fixed-chores-01 | cosine=0.6557
Paragraphs: chores-01, chores-02
The kitchen closer takes the evening handoff after the last meal. The schedule assigns one closer for each evening and lists a backup for weeks when a resident travels. Before starting, the closer checks the task board for notes about items soaking or food waiting to cool. A note should identify the item and the resident who will finish it. The closer can complete the rest of the routine while that resident handles the exception. Clean cookware belongs in the drying rack until it is dry enough to put away. Keep the main counter available for breakfast preparation. When the evening kitchen close is finished, mark the existing Kitchen Close card complete on the household task board. If the dishwasher is running, clip the Running tag to its handle

## Paired scaling comparison

In [5]:
# Recompute summary from hash-validated reviews, preserving raw outputs.
export_results(RUN_DIR)
display(pd.read_csv(RUN_DIR/'scaling_evaluation.csv')[['dataset','query_id','retrieved_context_quality',
    'answer_quality','irrelevant_chunk_fraction','retrieval_ms','generation_ms','end_to_end_ms','notes']])
display(pd.read_csv(RUN_DIR/'scaling_summary.csv'))
display(pd.read_csv(RUN_DIR/'scaling_pairs.csv'))
display(Markdown((ROOT/'a8/person2/FINDINGS.md').read_text()))

,dataset,query_id,retrieved_context_quality,answer_quality,irrelevant_chunk_fraction,retrieval_ms,generation_ms,end_to_end_ms,notes
0,small,Q1,100.0,0.0,0.333333,35.797030,21575.341641,21611.266888,"Credits completing the existing card. Invents marking dishwasher status on the Maintenance board and contacting the coordinator for repairs, instead of tagging the handle and deferring unloading. The retrieved tap-report passage is unrelated to normal dishwasher status."
1,small,Q2,100.0,0.0,0.333333,40.145317,12551.118736,12591.351337,"Out of stock note earns fact 2. Checked out is not unchecked and is treated as a contradictory status instruction, costing one point; fact 1 is not awarded. This wording judgment is explicit for student review. Post-trip outstanding-item context is relevant; monthly tool inventory at rank 3 is not."
2,small,Q3,100.0,0.0,0.666667,39.254688,35927.502920,35966.847196,"The normal blue hallway bin earns the location fact. The borrower name and complete expected-return record, drill-and-bits return and entry closure are not all conveyed. Invents a location for the checkout card and a green kitchen tool bin for the kit. Monthly inventory checks and guest-room supplies are irrelevant to the requested actions."
3,small,Q4,100.0,75.0,0.333333,32.831625,42192.313389,42225.305105,"All report details, existing-card reuse and through-card notification are correct; Plumbing and Maintenance board names are absent. Five minutes ago and yes are explicitly examples, not asserted observations. Kitchen morning routine at rank 3 is unrelated."
4,small,Q5,100.0,100.0,0.333333,49.610605,42861.613197,42911.330076,"All required quantities, placements, lamp check and completion condition appear before truncation of an optional final explanation. Guest-supply guidance at rank 2 is applicable; kitchen morning routine at rank 3 is unrelated."
5,full,Q1,100.0,0.0,0.333333,32.583845,21924.268021,21956.973924,Credits completing the closing card. Invents an appliance-maintenance status card and a technician service call; neither follows from a normally running dishwasher. Omits tag attachment and next-morning unloading. Rank 3 appliance upkeep is irrelevant.
6,full,Q2,100.0,0.0,0.333333,38.908397,13272.250341,13311.250360,Describes a purchase despite no approved replacement; neither required action is present. Refund context at rank 2 is irrelevant. Rank 3 inventory approval condition is applicable even though it contains no reference answer quote.
7,full,Q3,100.0,25.0,0.666667,38.288567,31778.610197,31816.993325,"Names the correct blue hallway bin and returns the kit there; kit is accepted as drill plus bits, consistently with the shared review. Omits borrower name and closing the entry. Directs lookup to the checkout card instead of identifying the home location directly. Monthly inventory review and appliance upkeep at ranks 2/3 do not answer the current checkout/return actions."
8,full,Q4,100.0,75.0,0.333333,31.130162,41625.889791,41657.134826,"All report details, existing-card reuse and through-card notification are correct; Plumbing and Maintenance board names are absent. Five minutes ago and yes are explicitly examples, not asserted observations. Kitchen morning routine at rank 3 is unrelated."
9,full,Q5,100.0,100.0,0.333333,35.846120,48646.321702,48682.277438,All required preparation facts appear before the cap cuts an optional concluding sentence. Rank 2 arrival/departure coordination is irrelevant; rank 3 explains optional supplies versus the required readiness items and is applicable context.


,dataset,queries,mean_context_quality,mean_answer_quality,mean_irrelevant_chunk_fraction,median_retrieval_ms,median_generation_ms,median_end_to_end_ms
0,small,5,100.0,35.0,0.4,39.254688,35927.502920,35966.847196
1,full,5,100.0,40.0,0.4,35.846120,31778.610197,31816.993325


,query_id,small_answer_quality,full_answer_quality,answer_quality_delta,small_noise_fraction,full_noise_fraction,small_end_to_end_ms,full_end_to_end_ms
0,Q1,0.0,0.0,0.0,0.333333,0.333333,21611.266888,21956.973924
1,Q2,0.0,0.0,0.0,0.333333,0.333333,12591.351337,13311.250360
2,Q3,0.0,25.0,25.0,0.666667,0.666667,35966.847196,31816.993325
3,Q4,75.0,75.0,0.0,0.333333,0.333333,42225.305105,41657.134826
4,Q5,100.0,100.0,0.0,0.333333,0.333333,42911.330076,48682.277438


# Person 2 — Assignment 8 results

Completed November 9, 2026 using the returned Google Colab CPU run. All 15 small-model matrix requests and 10 scaling requests contain actual answers, full retrieved chunks, exact prompts and measured end-to-end latency. Quality and relevance judgments are AI-assisted; student signoff is not recorded.

## Configuration and evidence

- Encoder: `sentence-transformers/all-MiniLM-L6-v2`, revision `1110a243fdf4706b3f48f1d95db1a4f5529b4d41`, 384 dimensions, normalized float32 cosine search, k=3.
- Generator: `Qwen/Qwen2-0.5B-Instruct`, revision `c540970f9e29518b1d8f06ab8b24cba66ad77b6d`, previously evaluated in A6/A7. The unchanged Person 3 system prompt, cited-context format, greedy decoding, 128-token cap and CPU float32 calculation are used throughout.
- Runtime: hosted Linux x86-64, Python 3.13.18, torch 2.14.1+cpu, NumPy 2.4.6, sentence-transformers 5.1.2, transformers 4.57.6, four PyTorch threads. Matrix/scaling manifests match on controls and environment.
- Full corpus: 30 paragraphs, 2,901 words. Small subset: 15 paragraphs, 1,461 words. The shared source/chunkers are unchanged. Both datasets retain all five reference answers.
- Raw evidence: `../results/person2/completed-2026-11-09/`; every review is SHA-256-bound to its entire raw row, including prompt, answer, hits and timing. All three downloaded ZIP snapshots agree on shared raw files; the executed Colab notebook contains no error outputs.

## Evaluation method

Context quality is the shared rubric’s percentage of reference quotes present in the top three retrieved texts. Source recall and reciprocal rank are separate diagnostics, not semantic precision. Answer quality awards each fully conveyed required fact, subtracts one fact point per distinct materially incorrect or unsupported instruction, floors at zero, then divides by the number of required facts. Compound facts require all parts. Exact fact indices, deductions and rationale are in `reviews.csv`.

Noise is separately reviewed: a chunk is relevant if it answers a requested part or supplies directly applicable conditions, handoff or approval context. A broad household vocabulary match is insufficient. Redundant but relevant passages are not automatically noise; a different source ID is not automatically noise. Noise fraction is `(3 - relevant_hits) / 3`.

## Fifteen-row evaluation table

All rows below use the MiniLM encoder and fixed Qwen generator. Full questions, answers, complete retrieved chunks and longer review notes are in `matrix_evaluation.csv`; raw JSONL also retains exact prompts. Latency is observed warmed request time from query encoding through decoded text, excluding downloads, index construction, warm-up and review.

| Strategy | Query | Context /100 | Answer /100 | Request seconds | Notes |
|---|---|---:|---:|---:|---|
| fixed | Q1 | 100 | 66.67 | 29.95 | Tag attachment omitted |
| fixed | Q2 | 100 | 0.00 | 11.52 | Incorrectly closes unpurchased entry |
| fixed | Q3 | 100 | 0.00 | 36.34 | Missing facts; invented log fields |
| fixed | Q4 | 100 | 0.00 | 40.02 | Conditional notification wrong; capped |
| fixed | Q5 | 100 | 20.00 | 25.93 | Missing placements/lamp/completion rule |
| overlap | Q1 | 100 | 100.00 | 41.11 | All handoff facts |
| overlap | Q2 | 100 | 0.00 | 14.65 | Quantity alone does not answer question |
| overlap | Q3 | 100 | 0.00 | 33.55 | Misuses appliance upkeep as drill context |
| overlap | Q4 | 100 | 75.00 | 37.00 | Omits Plumbing/Maintenance names |
| overlap | Q5 | 100 | 100.00 | 45.55 | All required facts before cap |
| hybrid | Q1 | 100 | 0.00 | 21.84 | Invents status card and service call |
| hybrid | Q2 | 100 | 0.00 | 13.89 | Describes a purchase despite no substitute |
| hybrid | Q3 | 100 | 25.00 | 31.52 | Correct kit bin; missing name/closure |
| hybrid | Q4 | 100 | 75.00 | 40.41 | Omits Plumbing/Maintenance names |
| hybrid | Q5 | 100 | 100.00 | 41.37 | All required facts before cap |

Query mapping: Q1 kitchen close/running dishwasher; Q2 unavailable grocery/no approved replacement; Q3 drill location/checkout/return; Q4 slow-drip reporting; Q5 guest-room readiness. The five verbatim queries and required facts remain in `../data/queries.json`.

| Strategy | Mean context | Mean answer | Median request seconds |
|---|---:|---:|---:|
| fixed | 100.00 | 17.33 | 29.95 |
| overlap | 100.00 | 55.00 | 37.00 |
| hybrid | 100.00 | 40.00 | 31.52 |

All 15 contexts contain every reference quote, yet the mean answer score is only 37.44. Retrieval availability does not guarantee the generator uses the evidence. Overlap performs best on this small-model slice; this is not the team-wide chunking comparison. Three matrix requests reach the output cap; the guest answers contain all required facts before the cut, whereas the fixed tap answer is incomplete. Score the text actually returned, not a presumed continuation.

## Controlled dataset scaling

Only dataset content changes within this experiment. Small model, hybrid chunking, k=3, queries, generator, prompt, decoding, cap, thread setting and software environment stay fixed. Small/full hybrid indexes have 10/20 chunks. Their raw float32 vector payloads are 15,360/30,720 bytes, calculated as chunks × 384 × 4; these exclude metadata, text, compression and model weights.

| Dataset | Evidence quality | Mean answer | Reviewed noise | Median retrieval ms | Median generation s | Median full request s |
|---|---:|---:|---:|---:|---:|---:|
| small | 100 | 35.00 | 40% | 39.25 | 35.93 | 35.97 |
| full | 100 | 40.00 | 40% | 35.85 | 31.78 | 31.82 |

| Query | Small answer /100 | Full answer /100 | Difference | Small/full noise |
|---|---:|---:|---:|---|
| Q1 | 0.00 | 0.00 | +0.00 | 33% / 33% |
| Q2 | 0.00 | 0.00 | +0.00 | 33% / 33% |
| Q3 | 0.00 | 25.00 | +25.00 | 67% / 67% |
| Q4 | 75.00 | 75.00 | +0.00 | 33% / 33% |
| Q5 | 100.00 | 100.00 | +0.00 | 33% / 33% |

**Retrieval quality:** both sizes keep the supporting source at rank one and all reference evidence within the top three. Added paragraphs change lower-ranked hits for Q1, Q2, Q3 and Q5; Q4 keeps the same hits. There is no evidence-coverage loss from this controlled distractor addition.

**Noise:** the reviewed irrelevant fraction stays 40% overall. Extra topics sometimes replace existing irrelevant context rather than adding more irrelevant hits at fixed k. The inventory substitution-approval clause is relevant to Q2 even though it supplies no answer quote; the refund passage is not. Q5’s guest-supply guidance is relevant, whereas arrival/departure coordination does not establish readiness. These judgments are disclosed and may be revised after student review.

**Answer quality:** the five-point mean increase comes entirely from Q3 (0→25): the small-set answer invents a green kitchen bin for the kit; the full-set answer returns the kit to the correct blue hallway bin but still omits the borrower name and entry closure. Both Q1 answers invent repair instructions despite complete handoff evidence. Larger data did not eliminate generation errors.

**Scoring sensitivity:** the small-set Q2 says “checked out” rather than “unchecked.” The base review credits Out of stock, withholds the correct-status fact and deducts one point for the contradictory status instruction, yielding zero. If a reviewer treats that phrase only as an omission and removes the deduction, this row becomes 50 and the small-set mean becomes 45 rather than 35. That reverses the aggregate ordering; the observed five-point advantage is not a robust general scaling benefit.

**Latency:** the full set is not consistently faster per query. These are five warmed requests per dataset, run in dataset order rather than a randomized repeated load test. Variable output length and hosted CPU conditions affect generation; generation dominates this tiny exact-search index. Small changes in median retrieval latency do not establish a scaling speed advantage. No 10K-user/day throughput claim follows.

## Integration and limitations

- These 25 rows complete the Person 2 experiment slice. Person 3’s separate 45-row comparison remains the matched cross-encoder evidence; do not pool Colab and Mac timing medians.
- Person 4 owns the combined chunking comparison, failure/improvement section, cost/design reflection and final team ZIP. The observed unsupported service call, out-of-stock failures and invented kit location are available for that analysis.
- The original 2026-11-08 Mac retrieval-only measurements and indexes remain untouched. See `RETRIEVAL_FINDINGS_2026-11-08.md`; their timings are not relabeled as Colab measurements.
- Exact prompts confirm no answer key entered generation. There is no context truncation. Five Colab requests in total reach the 128-token cap (three matrix, two scaling).
- Quality/relevance reviews are AI-assisted, not independent human adjudication. Student signoff is not recorded.


## Reproduction and integration

See `person2/README.md` for the Colab inference path, dependencies and review format. Defaults analyze saved evidence offline. A fresh inference run must use a fresh output directory and new hash-bound reviews; generated outputs may change across Python/PyTorch/hardware environments.

The smaller corpus has 15 paragraphs and the full corpus has 30; both retain all five reference answers. Only corpus size changes inside scaling. Timings from this Colab CPU run must not be pooled with Person 3's separate machine measurements. No conclusion about 10K users/day throughput follows from five queries on this tiny index.

Person 3 owns the pooled embedding-model comparison. Person 4 owns chunking/failure analysis, design reflection and final submission packaging.